# GRU Model for Battery RUL

This notebook starts from the shared window file created by preprocessing. The goal is to keep GRU-specific work separate from the shared data pipeline so GRU and GLU can be compared fairly.

In [ ]:
# Import only the tools needed to locate and inspect the prepared window data.
# We do not import PyTorch yet because the first step is to confirm the data shape.
from pathlib import Path

import numpy as np

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "processed" / "windows_w10.npz").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

WINDOW_FILE = PROJECT_ROOT / "data" / "processed" / "windows_w10.npz"

print("Project root:", PROJECT_ROOT)
print("Window file:", WINDOW_FILE)


In [ ]:
# Load the prepared windows so the GRU uses the same split and features as the GLU model.
# X contains sequences of battery cycles, and y contains the RUL at the last cycle of each sequence.
data = np.load(WINDOW_FILE, allow_pickle=True)

X_train = data["X_train"]
y_train = data["y_train"]
X_val = data["X_val"]
y_val = data["y_val"]
X_test = data["X_test"]
y_test = data["y_test"]

feature_columns = data["feature_columns"].tolist()
window_size = int(data["window_size"])

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Features:", feature_columns)
print("Window size:", window_size)


In [ ]:
# Validate the expected dimensions before model building.
# A GRU expects data shaped as samples, time steps, and features.
expected_feature_count = len(feature_columns)

assert X_train.ndim == 3
assert X_val.ndim == 3
assert X_test.ndim == 3
assert y_train.ndim == 1
assert y_val.ndim == 1
assert y_test.ndim == 1

assert X_train.shape[1:] == (window_size, expected_feature_count)
assert X_val.shape[1:] == (window_size, expected_feature_count)
assert X_test.shape[1:] == (window_size, expected_feature_count)
assert X_train.shape[0] == y_train.shape[0]
assert X_val.shape[0] == y_val.shape[0]
assert X_test.shape[0] == y_test.shape[0]

print("Data shape validation passed.")
print("Train samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples:", X_test.shape[0])
